# Day I20 — 중재와 웨어러블 로봇: 임상 근거는 어디까지이고, 실시간 제어는 무엇이 다른가

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I20.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I20.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 한 방향 필터와 `filtfilt`의 지연

In [ ]:
import numpy as np
from scipy import signal

fs = 100                                   # 가상 shank 자이로 표본화 주파수(Hz)
rng = np.random.default_rng(20)

def shank_gyro(n_strides=20, T=1.1, fs=100):
    """가상 shank 시상면 각속도(deg/s)와 실제 IC·toe-off 시각. 주기마다 길이가 조금 다르다."""
    w, ic, to, t0 = [], [], [], 0.0
    for _ in range(n_strides):
        Ti = T * (1 + rng.normal(0, 0.03))
        ph = np.arange(0, Ti, 1 / fs) / Ti                    # 이 stride 안의 0–1 위상
        g = -15 * np.sin(2 * np.pi * ph)                       # stance 동안 느린 변화
        g += -90 * np.exp(-((ph - 0.58) / 0.03) ** 2)          # toe-off 직전 음의 골
        sw = (ph >= 0.62)
        g[sw] += 330 * np.sin(np.pi * (ph[sw] - 0.62) / 0.38) ** 1.5   # swing: 큰 양의 봉우리
        g += -140 * np.exp(-((ph - 0.015) / 0.012) ** 2)       # IC 직후 짧은 음의 골
        ic.append(t0); to.append(t0 + 0.62 * Ti)
        w.append(g); t0 += len(ph) / fs
    return np.concatenate(w), np.array(ic), np.array(to)

clean, ic_true, to_true = shank_gyro()
gyro = clean + rng.normal(0, 12, len(clean))               # 센서 잡음을 더한 "측정값"
sos = signal.butter(4, 6, fs=fs, output="sos")             # 4차 Butterworth 저역통과 6 Hz

causal = signal.sosfilt(sos, gyro)                          # lfilter와 같은 한 방향(과거만 사용)
zero_phase = signal.sosfiltfilt(sos, gyro)                  # filtfilt: 앞뒤로 두 번(미래도 사용)

def peak_times(x):
    """mid-swing 봉우리(가장 큰 양의 값) 시각을 stride마다 찾는다."""
    pk, _ = signal.find_peaks(x, height=150, distance=int(0.6 * fs))
    return pk / fs

ref = peak_times(clean)                                    # 잡음 없는 원 신호의 봉우리 = 정답
for name, x in [("필터 없음", gyro), ("한 방향(sosfilt)", causal), ("앞뒤(sosfiltfilt)", zero_phase)]:
    lag = (peak_times(x) - ref) * 1000
    print(f"{name:16s} mid-swing 봉우리 시각 오차: 평균 {lag.mean():6.1f} ms, SD {lag.std():5.1f} ms")

# 같은 필터의 주파수별 지연(group delay): 보행 신호의 주성분 근처에서
w, gd = signal.group_delay(signal.butter(4, 6, fs=fs), w=[1, 2, 3], fs=fs)
for f_, d in zip(w, gd):
    print(f"  {f_:.0f} Hz 성분의 지연 ≈ {1000 * d / fs:.0f} ms")

# 실시간이라면? 지금까지 받은 표본만으로 filtfilt를 매번 다시 돌리면 '끝 부분'이 매번 달라진다
now = int(5.0 * fs)
offline_value = zero_phase[now]
online_value = signal.sosfiltfilt(sos, gyro[: now + 1])[-1]
print(f"t=5.00 s의 값 — 전체 기록으로 filtfilt: {offline_value:6.1f} | 그 시점까지로 filtfilt: {online_value:6.1f} deg/s")

### 예시 2 — 표본 하나씩 처리하는 stance/swing 상태기계

예시 1의 셀을 먼저 실행합니다. 상태기계는 미래 표본을 쓰지 않습니다. stance에서 각속도가 +100 deg/s를 넘으면 swing으로, swing 봉우리를 지난 뒤 −40 deg/s 아래로 꺾이면 IC로 보고 stance로 돌아갑니다. 전환 직후 0.25초는 판정하지 않습니다(같은 사건을 두 번 세지 않게).

In [ ]:
# 예시 1의 셀을 먼저 실행해 clean, gyro, ic_true, to_true, sos, fs가 있어야 한다.
import numpy as np
from scipy import signal

class StanceSwingFSM:
    """표본을 하나씩 받아 stance/swing을 판정하는 상태기계(미래 표본을 쓰지 않는다)."""
    def __init__(self, sos, fs, on_thr=100.0, ic_thr=-40.0, refractory=0.25, use_filter=True):
        self.sos, self.fs, self.use_filter = sos, fs, use_filter
        self.zi = signal.sosfilt_zi(sos) * 0.0          # 필터 내부 상태(처음엔 0)
        self.on_thr, self.ic_thr, self.refr = on_thr, ic_thr, refractory
        self.state, self.t_last, self.n = "STANCE", -1.0, 0
        self.peak = 0.0
        self.events = []                                   # (시각, "swing" 또는 "IC")

    def update(self, x):
        t = self.n / self.fs; self.n += 1
        if self.use_filter:
            y, self.zi = signal.sosfilt(self.sos, [x], zi=self.zi)   # 한 표본만 거른다
            x = y[0]
        if self.state == "SWING":
            self.peak = max(self.peak, x)                  # swing 중 가장 큰 각속도를 기억
        if t - self.t_last < self.refr:                   # 전환 직후 잠깐은 판정하지 않는다
            return self.state
        if self.state == "STANCE" and x > self.on_thr:     # 큰 양의 각속도 → swing 진행 중
            self.state, self.t_last, self.peak = "SWING", t, x
            self.events.append((t, "swing"))
        elif self.state == "SWING":
            if self.peak > 200 and x < self.ic_thr:          # 봉우리를 지나 음으로 꺾이면 IC
                self.state, self.t_last = "STANCE", t
                self.events.append((t, "IC"))
        return self.state

def evaluate(fsm_events, true_times, kind, tol=0.3):
    det = np.array([t for t, k in fsm_events if k == kind])
    err, used = [], set()
    for tt in true_times[1:]:                              # 첫 stride는 필터가 자리 잡는 구간이라 뺀다
        cand = det[(det >= tt - 0.05) & (det <= tt + tol)]
        if len(cand):
            err.append(cand[0] - tt); used.add(cand[0])
    extra = sum(1 for d in det if d not in used and d > true_times[1] - 0.05)
    return 1000 * np.mean(err), 1000 * np.std(err), len(err), len(true_times) - 1, extra

noisy = clean + np.random.default_rng(7).normal(0, 40, len(clean))   # 잡음이 큰 센서를 가정
for name, data, use_f in [("잡음 작음, 필터 없음", gyro, False), ("잡음 작음, 한 방향 6 Hz", gyro, True),
                          ("잡음 큼, 필터 없음", noisy, False), ("잡음 큼, 한 방향 6 Hz", noisy, True)]:
    fsm = StanceSwingFSM(sos, fs, use_filter=use_f)
    states = [fsm.update(v) for v in data]                 # 스트림처럼 한 표본씩
    m_to = evaluate(fsm.events, to_true, "swing")
    m_ic = evaluate(fsm.events, ic_true, "IC")
    print(f"[{name}] swing 판정: 실제 toe-off보다 평균 {m_to[0]:.0f} ms 늦음(SD {m_to[1]:.0f}), "
          f"{m_to[2]}/{m_to[3]} 검출, 여분 {m_to[4]}회")
    print(f"[{name}] IC 판정    : 실제 IC보다 평균 {m_ic[0]:.0f} ms 늦음(SD {m_ic[1]:.0f}), "
          f"{m_ic[2]}/{m_ic[3]} 검출, 여분 {m_ic[4]}회")

### 예시 3 — 두 가지 위상각: 각도–각속도(오프라인) 대 각도–시간적분(실시간)

가상 허벅지 각도로 두 방법을 비교합니다. (A)는 [RM] 13장처럼 주기 전체의 최솟값·최댓값으로 정규화하므로 **그 주기가 끝나야** 계산할 수 있습니다. (B)는 [R] 24장의 발상을 단순화한 것으로, 지난 주기의 평균과 배율만 쓰고 적분은 heel strike마다 0으로 되돌립니다. 여기서는 heel strike 시각을 정답으로 주었습니다(실제로는 예시 2 같은 검출기나 접지 센서가 필요).

In [ ]:
import numpy as np

fs = 100
rng = np.random.default_rng(21)

# 가상 허벅지(thigh) 각도(°): IC 무렵 굴곡 최대, stance 후반 신전 최대인 코사인 비슷한 모양 + 2배음
T_list = np.r_[np.full(10, 1.10), np.linspace(1.10, 0.90, 5), np.full(10, 0.90)]   # 중간에 빨라진다
pieces, phases, hs, n = [], [], [], 0
for T in T_list:
    ph = np.arange(0, T, 1 / fs) / T                 # 이 stride의 실제 위상 0→1
    pieces.append(5 + 20 * np.cos(2 * np.pi * ph) + 3 * np.cos(4 * np.pi * ph - 0.8))
    phases.append(ph); hs.append(n); n += len(ph)
clean, true_phase, hs = np.concatenate(pieces), np.concatenate(phases), np.array(hs)  # hs = heel strike 표본 번호
noise = rng.normal(0, 1, len(clean))
ends = np.r_[hs[1:], len(clean)]

def wrap_err(est, true):
    """원형 오차(주기 단위): 0.98과 0.02의 차이는 0.04다."""
    return (est - true + 0.5) % 1 - 0.5

def compare(sd):
    theta = clean + sd * noise                        # 각도 측정 잡음 sd(°)
    # (A) 각도–각속도 평면([RM] 13장 방식): 그 주기 전체의 min/max로 정규화 → 오프라인
    omega = np.gradient(theta, 1 / fs)                # 각속도(deg/s) — 미분은 잡음을 키운다
    phiA = np.zeros_like(theta)
    for a, b in zip(hs, ends):
        th, om = theta[a:b], omega[a:b]
        thn = 2 * (th - th.min()) / (th.max() - th.min()) - 1      # −1 … +1
        omn = om / np.abs(om).max()
        phiA[a:b] = np.arctan2(omn, thn)              # 이 평면에서는 시간이 가면 각이 '줄어든다'
    phaseA = (-phiA / (2 * np.pi)) % 1                # 부호를 뒤집어 0→1로 늘게

    # (B) 각도–시간적분 평면([R] 24장 방식): 지난 주기 정보만 쓰므로 실시간 계산 가능
    phaseB = np.full_like(theta, np.nan)
    offset, k = None, None
    for a, b in zip(hs, ends):
        if offset is not None:
            thc = theta[a:b] - offset                 # 지난 주기 평균을 빼 적분이 0 근처로 끝나게
            integ = np.cumsum(thc) / fs               # heel strike마다 적분을 0으로 리셋
            if k is not None:
                phaseB[a:b] = (np.arctan2(k * integ, thc) % (2 * np.pi)) / (2 * np.pi)
            k = np.ptp(thc) / np.ptp(integ)           # 다음 주기에 쓸 배율(두 축의 크기 맞추기)
        offset = theta[a:b].mean()

    ok = np.arange(len(theta)) >= hs[2]              # 두 방법 모두 셋째 주기부터 비교
    for name, est in [("A 각도–각속도, 오프라인 정규화", phaseA), ("B 각도–적분, 실시간", phaseB)]:
        e = wrap_err(est[ok], true_phase[ok])
        e = e - np.median(e)                          # 방법마다 다른 고정 오프셋은 빼고 '모양'만 비교
        print(f"잡음 {sd}° | {name}: 위상 오차 RMS {100 * np.sqrt(np.mean(e ** 2)):.1f} %GC, "
              f"최대 {100 * np.abs(e).max():.1f} %GC")

for sd in [0.0, 0.3, 1.0]:
    compare(sd)